# LightGBM Feature Importance Debugger

This notebook uses the training pipeline in **sampling mode** (25% stratified subset) to quickly train the LightGBM model and visualize the exact contribution of every feature into the decision-making process.

In [ ]:
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import lightgbm as lgb

# Add the current workspace directory to sys.path so we can import internal modules
sys.path.append(os.path.abspath('.'))

from pipeline import run_training_pipeline
import config

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 10)

## 1. Train the Model on 25% Sample
We use the `--sample` flag equivalent `(sample=True)` to quickly run the pipeline. This executes: Data Loading -> Engineering -> Splitting (25%) -> SMOTE -> LightGBM.

In [ ]:
# Set output paths so the pipeline writes here
print("Starting LightGBM pipeline on 25% stratified partition...")

# Run the pipeline with sample=True to use only 25% of the data uniformly stratified
metrics = run_training_pipeline(tune=False, sample=True)

print("\n--- Pipeline Finished. Model saved explicitly ---")

## 2. Load the Saved Model and Artifacts
We load `model.joblib` which contains the latest trained artifact and the preprocessing setup.

In [ ]:
import joblib
from preprocessing import preprocess
from data_loader import load_prime_data, load_transaction_data, merge_data
from feature_engineering import engineer_prime_features, engineer_transaction_features, create_target

# Load the model artifact specifically to inspect its inner workings
try:
    pipeline_data = joblib.load(config.MODEL_PATH)
    model = pipeline_data["model"]
    artifacts = pipeline_data["artifacts"]
    print("Loaded model and artifacts successfully.")
except Exception as e:
    print(f"Could not load the model from {config.MODEL_PATH}: {e}")

## 3. Retrieve Feature Names from Preprocessor
Since our data goes through `ColumnTransformer` (scaling and OneHotEncoding), the feature names change from their raw DataFrame headers. We must extract the actual encoded output headers.

In [ ]:
preprocessor = artifacts["preprocessor"]

try:
    # Get feature names mapped to the model's exact input shape
    feature_names = preprocessor.get_feature_names_out()
except Exception as e:
    print(f"Could not extract feature names: {e}")
    feature_names = [f"feature_{i}" for i in range(model.num_feature())]

print(f"Total features fed to LightGBM: {len(feature_names)}")

## 4. Feature Importance Visualization
We can measure the contribution to decisions using `split` importance (how many times the feature was used to split data) or `gain` importance (the total reduction in loss courtesy of splits that used the feature).

In [ ]:
if hasattr(model, 'feature_importance'):
    # importance_type='gain' shows how much *value* a feature brings to the network (its effectiveness)
    # importance_type='split' shows how *often* a feature is used (how many total times a tree divided using it)
    
    importance_gain = model.feature_importance(importance_type='gain')
    importance_split = model.feature_importance(importance_type='split')
    
    df_imp = pd.DataFrame({
        'Feature Name': feature_names,
        'Importance (Gain)': importance_gain,
        'Importance (Split)': importance_split
    })

    # Clean the column names outputted by sklearn pipeline (e.g. "num__utilization" -> "utilization")
    df_imp['Feature Name'] = df_imp['Feature Name'].str.replace('num__', '').str.replace('cat__', '')
    
    # Sort by the most impactful features (Gain)
    df_imp = df_imp.sort_values(by='Importance (Gain)', ascending=False).head(30) # Top 30 features

    plt.figure(figsize=(12, 12))
    sns.barplot(x='Importance (Gain)', y='Feature Name', data=df_imp, palette='viridis')
    plt.title('Top 30 Feature Importances (Gain - How much the feature reduced tree loss)', fontsize=14)
    plt.xlabel('Gain Importance', fontsize=12)
    plt.ylabel('Engineered Feature', fontsize=12)
    plt.tight_layout()
    plt.show()
    
    # Show split optionally
    # plt.figure(figsize=(12, 12))
    # sns.barplot(x='Importance (Split)', y='Feature Name', data=df_imp.sort_values(by='Importance (Split)', ascending=False).head(30), palette='magma')
    # plt.title('Top 30 Feature Importances (Split - How many times feature was used for decision)', fontsize=14)
    # plt.show()
    
else:
    print("Model doesn't support feature importance.")